# Reverse Diffusion: Noise to Data

Chapter 19: Stochastic Differential Equations · *Mathematics for AI and Machine Learning* by Fuheng Wu

Run the reverse-time SDE (or probability flow ODE) with the exact score from noise back to a two-mode distribution.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-19/reverse-diffusion/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import torch

torch.manual_seed(0)
mu, s, w = 2.0, 0.3, 0.7                 # data: w N(+mu, s^2) + (1-w) N(-mu, s^2)
T, steps, n = 3.0, 300, 20000


def score(x, t):
    # Exact score of p_t under the forward OU SDE dX = -X dt + sqrt(2) dW
    e = torch.exp(torch.tensor(-t))
    v = s**2 * e**2 + 1 - e**2
    means = torch.tensor([mu, -mu]) * e
    logw = torch.log(torch.tensor([w, 1 - w])) - (x[:, None] - means) ** 2 / (2 * v)
    post = torch.softmax(logw, dim=1)    # responsibilities of the two modes
    return (post * (-(x[:, None] - means) / v)).sum(dim=1)


# Reverse-time SDE dX = [f - G^2 score] dt + G dW_bar with f = -x, G = sqrt(2),
# integrated from t = T (pure noise) back to t = 0
dt = T / steps
x = torch.randn(n)
for i in range(steps):
    t = T - i * dt
    x = x + (x + 2 * score(x, t)) * dt + (2 * dt) ** 0.5 * torch.randn(n)

print("fraction at +mu mode:", round((x > 0).float().mean().item(), 3), " (data weight", w, ")")
print("mean of + mode:", round(x[x > 0].mean().item(), 3), " std:", round(x[x > 0].std().item(), 3))

fraction at +mu mode: 0.701  (data weight 0.7 )
mean of + mode: 1.999  std: 0.317
